# Python EDA
Moving over to Python to make some interactive charts maybe

In [1]:
import geopandas as gpd
states_shape = gpd.read_file("../data/cleandata/states_shifted.shp")

In [ ]:
#simplifying, getting rid of extra columns
states_shape = states_shape[['name', 'postal', 'latitude','longitude','geometry']]


In [4]:
import pandas as pd
norc_2024 = pd.read_csv('../data/rawdata/AP_NORC_survey.csv')

In [5]:
norc_2024[["state_abbr", "state_name"]] = norc_2024["P_STATE"].str.extract(r"\((.*?)\)\s*(.*)")

In [28]:
short_norc = norc_2024[['SU_ID', 'LIKELYVOTER', 'RACE0_VOTE','RACE0_PARTY','GENDER','TRACK','FORCAND','ABORTION',
'GUNPOLICY', 'VOTEFRAUD','FRACKING','FTVOTER','EDUC','INCOME','PARTYFULL','MARRIED',
'AGE65','RELIG4','ATTENDANCE','TIKTOKUSER','LGBT','state_abbr']]

#removing non-voters
short_norc = short_norc[short_norc["LIKELYVOTER"] != '(2) Nonvoter']
short_norc = short_norc[short_norc["AGE65"] != "(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)"]

### Age, political party, and gender

In [29]:
short_norc ['GENDER'] = short_norc['GENDER'].replace({'(2) Women': 'Female', '(1) Men': 'Male', '(3) Nonbinary/Some other way' : 'Other', "(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)" : 'Other'})

short_norc ['RACE0_VOTE'] = short_norc['RACE0_VOTE'].replace({'(8639) Donald Trump': 'Donald Trump', '(64984) Kamala Harris': 'Kamala Harris', '(100008) Another candidate' : 'Other', '(69459) Chase Oliver' : 'Other', ' ' : 'Other', '(895) Jill Stein' : 'Other', '(71841) Robert Kennedy' : 'Other','(71845) Cornel West' : 'Other', '(100013) None of These Candidates' : 'Other'})

In [30]:
total_voters = (
    short_norc.groupby(["AGE65","RACE0_VOTE"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

male_voters = (
    short_norc[short_norc["GENDER"]=="Male"]
    .groupby(["AGE65","RACE0_VOTE"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

female_voters = (
    short_norc[short_norc["GENDER"]=="Female"]
    .groupby(["AGE65","RACE0_VOTE"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

In [37]:
import plotly.graph_objects as go

fig = go.Figure()

#colors
trump_color = "#E9141D"
harris_color = "#0015BC" 
other_color = '#B0B0B0'

# TOTAL
fig.add_bar(x=total_voters.AGE65, y=total_voters["Donald Trump"], name="Donald Trump", visible=True, marker_color=trump_color)
fig.add_bar(x=total_voters.AGE65, y=total_voters["Kamala Harris"], name="Kamala Harris", visible=True, marker_color=harris_color)
fig.add_bar(x=total_voters.AGE65, y=total_voters["Other"], name="Other", visible=True, marker_color=other_color)

# MALE
fig.add_bar(x=male_voters.AGE65, y=male_voters["Donald Trump"], name="Donald Trump", visible=False, marker_color=trump_color)
fig.add_bar(x=male_voters.AGE65, y=male_voters["Kamala Harris"], name="Kamala Harris", visible=False, marker_color=harris_color)
fig.add_bar(x=male_voters.AGE65, y=male_voters["Other"], name="Other", visible=False, marker_color=other_color)

# FEMALE
fig.add_bar(x=female_voters.AGE65, y=female_voters["Donald Trump"], name="Donald Trump", visible=False, marker_color=trump_color)
fig.add_bar(x=female_voters.AGE65, y=female_voters["Kamala Harris"], name="Kamala Harris", visible=False, marker_color=harris_color)
fig.add_bar(x=female_voters.AGE65, y=female_voters["Other"], name="Other", visible=False, marker_color=other_color)

fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Total",
                    method="update",
                    
                    args=[{"visible":[True,True,True,False,False,False,False,False,False]}, {"title":"2024 Vote by Age Group (All Voters)"}]
                ),
                dict(
                    label="Male",
                    method="update",
                    args=[{"visible":[False,False,False,True,True,True,False,False,False]}, {"title":"2024 Vote by Age Group (Male Voters)"}]
                ),
                dict(
                    label="Female",
                    method="update",
                    args=[{"visible":[False,False,False,False,False,False,True,True,True]}, {"title":"2024 Vote by Age Group (Female Voters)"}]
                )
            ],
            direction="down"
        )
    ],
    barmode="group",
    xaxis_title="Age Group",
    yaxis_title="Number of Voters"
)

fig.update_layout(
    title="2024 Vote by Age Group (All Voters)"
)

fig.update_layout(
    yaxis=dict(range=[0, 20000])
)

fig.show()

In [ ]:
#plotly needs geojson, for some reason
#states_gjs = states_shape.__geo_interface__